## Plot GNSS Orbits

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import pylupnt as pnt
from tqdm import tqdm
from datetime import datetime, timedelta
from src.setup_ionoenv import setup_lcrns_sats, setup_gnss_constellation
from src.gnss_meas import GNSSMeas

## Settings

In [ ]:
gps_datetime = datetime(2025, 3, 1, 12, 0, 0)  # set the GPS datetime for the simulation
n_orbit = 6.0  # number of orbits to simulate for TDCP
dt = 1.0  # time step in seconds
tidx_inv_raytrace = 120  # time index for inverse raytracing (6 corresponds to 1 minute)
overwrite_rx_orbit = False  # whether to overwrite existing orbit data
overwrite_gnss_orbit = False  # whether to overwrite existing GNSS data
overwrite_gnss_measurements = False  # whether to overwrite existing GNSS measurements
overwrite_labels = False  # whether to overwrite existing ionosphere label data
consider_gnss_fault = True  # whether to consider GNSS faults
sp3_prop_method = (
    "interp"  # method for SP3 propagation ("interp" gives better accuracy)
)

### Generate user orbits

In [ ]:
t_tai, rv_m2sc_ci, rv_e2sc_ecef, N_sc = setup_lcrns_sats(
    n_orbit=n_orbit, dt=dt, savefig=False, overwrite=overwrite_rx_orbit
)

In [ ]:
outdir = pnt.get_output_dir() / "iono_delay" / "orbits" / "figures"

### PLot lunar satellite orbits

In [ ]:
# plot to verify
from plotly import graph_objects as go

figname = outdir / "lcrns_orbits.pdf"
fig = go.Figure()
pnt.plot.plot_orbits(fig, rv_m2sc_ci[0], color="black")  # [N, t, 3]
# get initial epoch Earth direction
earth_dir = -rv_e2sc_ecef[0, 0] / np.linalg.norm(rv_e2sc_ecef[0, 0])
print("Earth direction (ECEF):", earth_dir)
sun_pos = pnt.get_body_pos_vel(t_tai[0], pnt.EARTH, pnt.SUN, pnt.ECEF)[:3]
sun_dir = sun_pos / np.linalg.norm(sun_pos)
print("Sun direction (ECEF):", sun_dir)
#
pnt.plot.plot_arrow3(
    fig,
    origin=np.array([0, 0, 0]),
    direction=sun_dir,
    color="red",
    length=50,
    tip=20,
    width=3,
    scale=1,
)
pnt.plot.plot_arrow3(
    fig,
    origin=np.array([0, 0, 0]),
    direction=earth_dir,
    color="blue",
    length=50,
    tip=20,
    width=3,
    scale=1,
)
# pnt.plot.plot_orbits(fig, rv_pa_loaded, color="blue")
pnt.plot.plot_body(
    fig,
    pnt.MOON,
    size_factor=2,
    alpha=0.5,
)
pnt.plot.set_view(fig, -80, 18, 3.1)
fig.update_layout(showlegend=True, width=400, height=400)
fig.write_image(figname)
fig.show()

In [ ]:
import os

gnss_meas_dir = os.path.join(pnt.get_output_dir(), "iono_delay")

# setup GNSS constellation and measurements
gnss_meas = GNSSMeas(
    t_tai,
    rv_m2sc_ci,
    basepath=gnss_meas_dir,
    consider_faults=True,
    rv_e2sc_ecef=rv_e2sc_ecef,
)
gnss_meas.setup_gnss(
    gnss_consts=["GPS", "GALILEO", "QZSS"],
    gps_datetime=gps_datetime,
    overwrite=False,
    sp3_prop_method=sp3_prop_method,
)

In [ ]:
import matplotlib.pyplot as plt

gnss_figname = os.path.join(outdir, "gnss_orbits.pdf")

black_bg = True

fig = go.Figure()

pnt.plot.plot_body(fig, pnt.EARTH, size_factor=5)

orbit_colors = {
    "GPS": "blue",
    "GALILEO": "orange",
    "QZSS": "green",
}

# gnss
plot_inv = 100
for gnss_const in gnss_meas.gnss_consts:
    print("Plotting {} orbits...".format(gnss_const))
    rv_eci = gnss_meas.rv_gnss_eci[gnss_const]
    tidx = np.arange(0, rv_eci.shape[1], plot_inv)
    pnt.plot.plot_orbits(fig, rv_eci[:, tidx, :], t=0, color=orbit_colors[gnss_const])

pnt.plot.set_view(fig, -80, 20, 2.8)
fig.update_layout(showlegend=True, width=400, height=400)

if black_bg:
    # make background black
    fig.update_layout(paper_bgcolor="black", plot_bgcolor="black")
    # delete axis
    fig.update_layout(
        scene=dict(
            xaxis=dict(
                showbackground=False,
                visible=False,
                showticklabels=False,
                showgrid=False,
            ),
            yaxis=dict(
                showbackground=False,
                visible=False,
                showticklabels=False,
                showgrid=False,
            ),
            zaxis=dict(
                showbackground=False,
                visible=False,
                showticklabels=False,
                showgrid=False,
            ),
        )
    )
    # delete labels
    fig.update_layout(scene=dict(xaxis_title="", yaxis_title="", zaxis_title=""))

fig.write_image(gnss_figname)
fig.show()